In [3]:
# =========================
# CELL 0: SETUP + SEEDS
# =========================
import os, random, gc, math
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    brier_score_loss, confusion_matrix
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ---------- Repro ----------
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ---------- Device ----------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ---------- Outputs ----------
OUT_DIR = "./outputs"
FIG_DIR = os.path.join(OUT_DIR, "figures")
TAB_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

def save_fig(path, dpi=300):
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()

print("Output dirs:", OUT_DIR, FIG_DIR, TAB_DIR)

# =========================
# CELL 1: CONFIG
# =========================
@dataclass
class CFG:
    seed: int = 42
    n_splits: int = 5
    shuffle: bool = True
    
    target_col: str = "stroke"   # based on your dataset
    id_cols: tuple = ()          # e.g., ("id",) if any
    
    # training defaults (we'll refine later)
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs: int = 30
    patience: int = 7
    
    # imbalance
    use_pos_weight: bool = True   # pos_weight for BCE
    use_focal: bool = True
    focal_gamma: float = 2.0
    
    # ensemble (we will set M later)
    ensemble_M: int = 3

cfg = CFG()
print(cfg)

# =========================
# CELL 2: LOAD DATA
# =========================
# Put your CSV path here.
# Examples:
# Kaggle: "/kaggle/input/<dataset-folder>/stroke.csv"
# Local upload: "/kaggle/working/stroke.csv"
DATA_PATH = "/kaggle/input/stroke-dataset/Stroke.csv"

assert os.path.exists(DATA_PATH), f"File not found at: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

# =========================
# CELL 3: AUDIT
# =========================
assert cfg.target_col in df.columns, f"Target column '{cfg.target_col}' not in columns."

# Drop ID-like columns if needed
for c in cfg.id_cols:
    if c in df.columns:
        df = df.drop(columns=[c])

y = df[cfg.target_col].astype(int).values
X = df.drop(columns=[cfg.target_col])

pos_rate = y.mean()
print(f"N={len(y)} | Positives={y.sum()} | PosRate={pos_rate:.4f}")

# Missingness
miss = X.isna().mean().sort_values(ascending=False)
miss_top = miss.head(20)
display(pd.DataFrame({"missing_rate": miss_top}))

# Quick class plot
plt.figure()
plt.bar(["0 (no stroke)", "1 (stroke)"], [np.sum(y==0), np.sum(y==1)])
plt.title("Class distribution")
save_fig(os.path.join(FIG_DIR, "class_distribution.png"))
print("Saved:", os.path.join(FIG_DIR, "class_distribution.png"))

# =========================
# CELL 4: COLUMN TYPES
# =========================
# Auto-detect categorical columns:
# - object / category
# - or low-cardinality integers (optional)
X_ = X.copy()

cat_cols = X_.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_.columns if c not in cat_cols]

# Optional: treat low-cardinality int as categorical (uncomment if needed)
# for c in num_cols.copy():
#     if pd.api.types.is_integer_dtype(X_[c]) and X_[c].nunique(dropna=True) <= 15:
#         cat_cols.append(c)
#         num_cols.remove(c)

print("n_cat:", len(cat_cols), "n_num:", len(num_cols))
print("cat sample:", cat_cols[:10])
print("num sample:", num_cols[:10])

# Save schema for reproducibility
schema = {
    "target": cfg.target_col,
    "cat_cols": cat_cols,
    "num_cols": num_cols
}
pd.Series(schema, dtype="object").to_json(os.path.join(OUT_DIR, "schema.json"))
print("Saved schema:", os.path.join(OUT_DIR, "schema.json"))

# =========================
# CELL 5: PREPROCESS PIPELINE (SKLEARN)
# =========================
num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop"
)

# quick fit-transform sanity (not for training yet)
Xt = preprocess.fit_transform(X_)
print("Transformed shape:", Xt.shape)

# =========================
# CELL 6: SPLITS + BASELINE (LogReg)
# =========================
from sklearn.linear_model import LogisticRegression

def eval_binary(y_true, p):
    # p = probability of class 1
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)
    pr = average_precision_score(y_true, p)
    roc = roc_auc_score(y_true, p)
    brier = brier_score_loss(y_true, p)
    return {"pr_auc": pr, "roc_auc": roc, "brier": brier}

skf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=cfg.shuffle, random_state=cfg.seed)

fold_metrics = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_, y), 1):
    X_tr, X_va = X_.iloc[tr_idx], X_.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    
    # Fit preprocessing on train only (leakage-safe)
    Xtr = preprocess.fit_transform(X_tr)
    Xva = preprocess.transform(X_va)
    
    # Baseline classifier (strong sanity check)
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=None)
    clf.fit(Xtr, y_tr)
    p_va = clf.predict_proba(Xva)[:, 1]
    
    m = eval_binary(y_va, p_va)
    m["fold"] = fold
    fold_metrics.append(m)
    print(f"[fold {fold}] PR={m['pr_auc']:.4f} ROC={m['roc_auc']:.4f} Brier={m['brier']:.4f}")

base_df = pd.DataFrame(fold_metrics)
display(base_df)

summary = base_df[["pr_auc", "roc_auc", "brier"]].agg(["mean", "std"])
display(summary)

# Save baseline table
base_df.to_csv(os.path.join(TAB_DIR, "baseline_logreg_cv.csv"), index=False)
summary.to_csv(os.path.join(TAB_DIR, "baseline_logreg_cv_summary.csv"))
print("Saved:", os.path.join(TAB_DIR, "baseline_logreg_cv.csv"))

DEVICE: cpu
Output dirs: ./outputs ./outputs/figures ./outputs/tables
CFG(seed=42, n_splits=5, shuffle=True, target_col='stroke', id_cols=(), batch_size=256, lr=0.001, weight_decay=1e-05, epochs=30, patience=7, use_pos_weight=True, use_focal=True, focal_gamma=2.0, ensemble_M=3)
Shape: (4603, 36)


,stroke,gender,age,Race,Marital status,alcohol,smoke,sleep disorder,Health Insurance,General health condition,...,energy,protein,Carbohydrate,Dietary fiber,Total fat,Total saturated fatty acids,Total monounsaturated fatty acids,Total polyunsaturated fatty acids,Potassium,Sodium
0,0,2,2,5,1,0,0,2,2,3,...,1598,62.78,192.19,10.0,65.64,25.112,24.090,8.543,2887,2969
1,0,2,2,1,1,0,0,1,2,3,...,1547,45.35,256.02,17.0,42.56,13.423,15.389,10.613,2058,2091
2,1,1,2,3,1,1,1,2,1,3,...,2466,81.56,254.49,13.0,103.32,43.295,36.727,15.366,3117,5233
3,0,2,3,3,1,1,1,2,1,4,...,1605,70.99,143.37,10.0,81.60,24.527,30.567,18.174,1766,3706
4,0,1,1,4,1,0,0,2,1,2,...,1818,74.75,229.45,14.2,67.49,26.030,24.837,10.533,1842,2461


N=4603 | Positives=362 | PosRate=0.0786


,missing_rate
gender,0.0
age,0.0
Race,0.0
Marital status,0.0
alcohol,0.0
smoke,0.0
sleep disorder,0.0
Health Insurance,0.0
General health condition,0.0
depression,0.0


Saved: ./outputs/figures/class_distribution.png
n_cat: 0 n_num: 35
cat sample: []
num sample: ['gender', 'age', 'Race', 'Marital status', 'alcohol ', 'smoke', 'sleep disorder', 'Health Insurance', 'General health condition', 'depression']
Saved schema: ./outputs/schema.json
Transformed shape: (4603, 35)
[fold 1] PR=0.2084 ROC=0.7282 Brier=0.2109
[fold 2] PR=0.1872 ROC=0.7187 Brier=0.2056
[fold 3] PR=0.1418 ROC=0.6999 Brier=0.2164
[fold 4] PR=0.1489 ROC=0.6748 Brier=0.2279
[fold 5] PR=0.1865 ROC=0.7334 Brier=0.2001


,pr_auc,roc_auc,brier,fold
0,0.208447,0.728193,0.210876,1
1,0.187232,0.718661,0.205645,2
2,0.141789,0.699922,0.216409,3
3,0.148905,0.674790,0.227921,4
4,0.186505,0.733425,0.200109,5


,pr_auc,roc_auc,brier
mean,0.174576,0.710998,0.212192
std,0.028212,0.023932,0.010675


Saved: ./outputs/tables/baseline_logreg_cv.csv


In [4]:
# =========================
# CELL 7: MANUAL COLUMN TYPES
# =========================

# Based on dataset structure (integer-coded categories)
cat_cols = [
    'gender',
    'Race',
    'Marital status',
    'alcohol ',
    'smoke',
    'sleep disorder',
    'Health Insurance',
    'General health condition',
    'depression',
    'diabetes',
    'hypertension',
    'high cholesterol',
    'Coronary Heart Disease'
]

# Remaining columns = numeric
num_cols = [c for c in X_.columns if c not in cat_cols]

print("n_cat:", len(cat_cols), "n_num:", len(num_cols))
print("Categorical:", cat_cols[:10])
print("Numeric:", num_cols[:10])

# =========================
# CELL 8: PREPROCESS (UPDATED)
# =========================

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

Xt = preprocess.fit_transform(X_)
print("New transformed shape:", Xt.shape)

# =========================
# CELL 9: STRONG BASELINES
# =========================

from sklearn.ensemble import RandomForestClassifier

try:
    import xgboost as xgb
    has_xgb = True
except:
    has_xgb = False

try:
    import lightgbm as lgb
    has_lgb = True
except:
    has_lgb = False


def run_model_cv(model, name):
    fold_metrics = []
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_, y), 1):
        X_tr, X_va = X_.iloc[tr_idx], X_.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        
        Xtr = preprocess.fit_transform(X_tr)
        Xva = preprocess.transform(X_va)
        
        model.fit(Xtr, y_tr)
        p_va = model.predict_proba(Xva)[:, 1]
        
        m = eval_binary(y_va, p_va)
        m["fold"] = fold
        fold_metrics.append(m)
    
    dfm = pd.DataFrame(fold_metrics)
    print(f"\n{name}")
    print(dfm[["pr_auc","roc_auc","brier"]].mean())
    return dfm


# ---------- Random Forest ----------
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight="balanced",
    random_state=cfg.seed,
    n_jobs=-1
)
rf_df = run_model_cv(rf, "RandomForest")

# ---------- XGBoost ----------
if has_xgb:
    xgb_model = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        scale_pos_weight=(len(y)-y.sum())/y.sum(),
        random_state=cfg.seed
    )
    xgb_df = run_model_cv(xgb_model, "XGBoost")

# ---------- LightGBM ----------
if has_lgb:
    lgb_model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=64,
        class_weight="balanced",
        random_state=cfg.seed
    )
    lgb_df = run_model_cv(lgb_model, "LightGBM")

n_cat: 13 n_num: 22
Categorical: ['gender', 'Race', 'Marital status', 'alcohol ', 'smoke', 'sleep disorder', 'Health Insurance', 'General health condition', 'depression', 'diabetes']
Numeric: ['age', 'sleep time', 'Minutes sedentary activity', 'Body Mass Index', 'Waist Circumference', 'Systolic blood pressure', 'Diastolic blood pressure', 'High-density lipoprotein', 'Triglyceride', 'Low-density lipoprotein']
New transformed shape: (4603, 59)

RandomForest
pr_auc     0.139212
roc_auc    0.674565
brier      0.070818
dtype: float64

XGBoost
pr_auc     0.128681
roc_auc    0.628493
brier      0.081874
dtype: float64
[LightGBM] [Info] Number of positive: 290, number of negative: 3392
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002859 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3982
[LightGBM] [Info] Number of data points in the train 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 289, number of negative: 3393
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000786 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3983
[LightGBM] [Info] Number of data points in the train set: 3682, number of used features: 59
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 289, number of negative: 3393
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000784 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3981
[LightGBM] [Info] Number of data points in the train set: 3682, number of used features: 59
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 290, number of negative: 3393
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000783 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3980
[LightGBM] [Info] Number of data points in the train set: 3683, number of used features: 59
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 290, number of negative: 3393
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000791 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3984
[LightGBM] [Info] Number of data points in the train set: 3683, number of used features: 59
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000

LightGBM
pr_auc     0.128036
roc_auc    0.643211
brier      0.079313
dtype: float64


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
# =========================
# CELL 10: TORCH DATASET
# =========================

import torch
from torch.utils.data import Dataset, DataLoader

class TabularDataset(Dataset):
    def __init__(self, X_cat, X_num, y):
        self.X_cat = torch.LongTensor(X_cat)
        self.X_num = torch.FloatTensor(X_num)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_cat[idx], self.X_num[idx], self.y[idx]

# =========================
# CELL 11: DL PREPROCESS
# =========================

# Encode categorical as integer codes (for embeddings)
X_cat_all = X_[cat_cols].copy()
for c in cat_cols:
    X_cat_all[c] = X_cat_all[c].astype("category").cat.codes + 1  # reserve 0 for missing/UNK

X_num_all = X_[num_cols].copy()
scaler = StandardScaler()
X_num_all = scaler.fit_transform(X_num_all)

X_cat_all = X_cat_all.values
X_num_all = X_num_all.astype(np.float32)

print("Cat shape:", X_cat_all.shape)
print("Num shape:", X_num_all.shape)

# =========================
# CELL 12: MODEL
# =========================

class TabularTransformer(nn.Module):
    def __init__(self, cat_dims, num_dim, d_model=64, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()

        # Categorical embeddings
        self.cat_embeds = nn.ModuleList([
            nn.Embedding(dim, d_model) for dim in cat_dims
        ])

        # Numeric projection
        self.num_proj = nn.Linear(num_dim, d_model)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model*4,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Output head
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

    def forward(self, x_cat, x_num):
        cat_tokens = [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embeds)]
        cat_tokens = torch.stack(cat_tokens, dim=1)   # (B, n_cat, d)

        num_token = self.num_proj(x_num).unsqueeze(1) # (B,1,d)

        tokens = torch.cat([cat_tokens, num_token], dim=1)
        h = self.encoder(tokens)

        out = h.mean(dim=1)
        logit = self.head(out).squeeze(1)
        return logit

# =========================
# CELL 13: LOSS
# =========================

class FocalBCE(nn.Module):
    def __init__(self, pos_weight=None, gamma=2.0):
        super().__init__()
        self.pos_weight = pos_weight
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, reduction='none',
            pos_weight=self.pos_weight
        )
        prob = torch.sigmoid(logits)
        pt = torch.where(targets == 1, prob, 1 - prob)
        loss = ((1 - pt) ** self.gamma) * bce
        return loss.mean()

Cat shape: (4603, 13)
Num shape: (4603, 22)


In [6]:
# =========================
# CELL 14: METRICS + ERROR ANALYSIS HELPERS
# =========================
from sklearn.metrics import precision_recall_curve, roc_curve

def to_np(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def eval_binary_full(y_true, p):
    p = np.clip(p, 1e-12, 1-1e-12)
    pr = average_precision_score(y_true, p)
    roc = roc_auc_score(y_true, p)
    brier = brier_score_loss(y_true, p)
    return pr, roc, brier

def threshold_report(y_true, p, thr):
    y_hat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()
    acc = (tp + tn) / (tp + tn + fp + fn)
    tpr = tp / (tp + fn + 1e-12)  # recall pos
    fpr = fp / (fp + tn + 1e-12)
    prec = tp / (tp + fp + 1e-12)
    bal_acc = 0.5 * (tpr + tn/(tn+fp+1e-12))
    return {
        "thr": thr, "acc": acc, "bal_acc": bal_acc,
        "precision": prec, "tpr": tpr, "fpr": fpr,
        "tp": tp, "fp": fp, "tn": tn, "fn": fn
    }

def best_threshold_by_f1(y_true, p):
    prec, rec, thr = precision_recall_curve(y_true, p)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    # precision_recall_curve returns thr with length-1
    best_i = np.argmax(f1[:-1]) if len(f1) > 1 else 0
    best_thr = thr[best_i] if len(thr) else 0.5
    return best_thr

# =========================
# CELL 15: BALANCED SAMPLER DATALOADER
# =========================
from torch.utils.data import WeightedRandomSampler

def make_loaders(X_cat, X_num, y, tr_idx, va_idx, batch_size=256):
    Xc_tr, Xc_va = X_cat[tr_idx], X_cat[va_idx]
    Xn_tr, Xn_va = X_num[tr_idx], X_num[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    ds_tr = TabularDataset(Xc_tr, Xn_tr, y_tr)
    ds_va = TabularDataset(Xc_va, Xn_va, y_va)

    # Weighted sampler to oversample positives
    ytr_t = torch.tensor(y_tr, dtype=torch.float32)
    w = torch.where(ytr_t == 1, torch.tensor(1.0), torch.tensor(1.0))
    # make positive weight higher
    pos = (y_tr == 1).sum()
    neg = (y_tr == 0).sum()
    pos_w = neg / max(pos, 1)  # typical
    w = torch.where(ytr_t == 1, torch.tensor(float(pos_w)), torch.tensor(1.0))
    sampler = WeightedRandomSampler(weights=w, num_samples=len(w), replacement=True)

    dl_tr = DataLoader(ds_tr, batch_size=batch_size, sampler=sampler, drop_last=False)
    dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False, drop_last=False)

    return dl_tr, dl_va, pos_w

# =========================
# CELL 16: TRAINING LOOP (EARLY STOP ON PR-AUC)
# =========================
def train_one_fold(X_cat, X_num, y, tr_idx, va_idx, fold, 
                   d_model=64, n_heads=4, n_layers=2, dropout=0.25,
                   lr=7e-4, weight_decay=1e-5, epochs=40, patience=8, batch_size=256):

    dl_tr, dl_va, pos_w = make_loaders(X_cat, X_num, y, tr_idx, va_idx, batch_size=batch_size)

    # categorical vocab sizes
    cat_dims = []
    for j in range(X_cat.shape[1]):
        cat_dims.append(int(X_cat[:, j].max()) + 1)  # +1 already in codes

    model = TabularTransformer(cat_dims=cat_dims, num_dim=X_num.shape[1],
                               d_model=d_model, n_heads=n_heads, n_layers=n_layers,
                               dropout=dropout).to(DEVICE)

    pos_weight = torch.tensor([pos_w], dtype=torch.float32, device=DEVICE)  # for BCE
    criterion = FocalBCE(pos_weight=pos_weight, gamma=cfg.focal_gamma).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_pr = -1
    best_state = None
    bad = 0
    history = []

    for ep in range(epochs):
        model.train()
        tr_losses = []
        for xb_cat, xb_num, yb in dl_tr:
            xb_cat = xb_cat.to(DEVICE)
            xb_num = xb_num.to(DEVICE)
            yb = yb.to(DEVICE)

            opt.zero_grad()
            logits = model(xb_cat, xb_num)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_losses.append(loss.item())

        # ---- validate
        model.eval()
        p_va_all = []
        y_va_all = []
        with torch.no_grad():
            for xb_cat, xb_num, yb in dl_va:
                xb_cat = xb_cat.to(DEVICE)
                xb_num = xb_num.to(DEVICE)
                logits = model(xb_cat, xb_num)
                p = torch.sigmoid(logits)
                p_va_all.append(to_np(p))
                y_va_all.append(to_np(yb))

        p_va = np.concatenate(p_va_all)
        y_va = np.concatenate(y_va_all).astype(int)

        pr, roc, brier = eval_binary_full(y_va, p_va)
        tr_loss = float(np.mean(tr_losses))
        history.append({"fold": fold, "epoch": ep, "tr_loss": tr_loss, "pr": pr, "roc": roc, "brier": brier})

        print(f"[fold {fold} ep {ep:02d}] loss={tr_loss:.4f} PR={pr:.4f} ROC={roc:.4f} Brier={brier:.4f}")

        # early stop on PR
        if pr > best_pr + 1e-4:
            best_pr = pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    return model, history, (y_va, p_va)

# =========================
# CELL 17: CV TRAIN (SINGLE MODEL PER FOLD)
# =========================
set_seed(cfg.seed)

all_hist = []
oof = np.zeros(len(y), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_, y), 1):
    model, hist, (y_va, p_va) = train_one_fold(
        X_cat_all, X_num_all, y, tr_idx, va_idx, fold,
        d_model=64, n_heads=4, n_layers=2, dropout=0.25,
        lr=7e-4, weight_decay=1e-5, epochs=40, patience=8,
        batch_size=256
    )
    all_hist.extend(hist)
    oof[va_idx] = p_va

# Overall
pr, roc, brier = eval_binary_full(y, oof)
print("\n==== OOF OVERALL ====")
print("PR-AUC:", pr)
print("ROC-AUC:", roc)
print("Brier:", brier)

hist_df = pd.DataFrame(all_hist)
hist_df.to_csv(os.path.join(TAB_DIR, "ft_transformer_cv_history.csv"), index=False)

# =========================
# CELL 18: MISTAKE ANALYSIS (OOF)
# =========================

# PR curve plot
prec, rec, thr = precision_recall_curve(y, oof)
plt.figure()
plt.plot(rec, prec)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("OOF Precision–Recall Curve")
save_fig(os.path.join(FIG_DIR, "oof_pr_curve.png"))
print("Saved:", os.path.join(FIG_DIR, "oof_pr_curve.png"))

best_thr = best_threshold_by_f1(y, oof)
rep = threshold_report(y, oof, best_thr)
print("\nBest-F1 threshold:", best_thr)
print(rep)

# Confusion breakdown
tn, fp, fn, tp = rep["tn"], rep["fp"], rep["fn"], rep["tp"]
print(f"\nConfusion @thr={best_thr:.3f}: TP={tp} FP={fp} TN={tn} FN={fn}")

# ---- Hard errors
df_err = X_.copy()
df_err["y"] = y
df_err["p"] = oof
df_err["pred"] = (oof >= best_thr).astype(int)

false_neg = df_err[(df_err["y"]==1) & (df_err["pred"]==0)].sort_values("p")
false_pos = df_err[(df_err["y"]==0) & (df_err["pred"]==1)].sort_values("p", ascending=False)

print("\nTop 10 hardest FALSE NEGATIVES (missed strokes):")
display(false_neg.head(10))

print("\nTop 10 highest-confidence FALSE POSITIVES:")
display(false_pos.head(10))

# ---- subgroup breakdown
def group_report(df, col):
    out = []
    for g, dfg in df.groupby(col):
        if len(dfg) < 50:
            continue
        prg, rocg, bg = eval_binary_full(dfg["y"].values, dfg["p"].values)
        thrg = best_threshold_by_f1(dfg["y"].values, dfg["p"].values)
        repg = threshold_report(dfg["y"].values, dfg["p"].values, thrg)
        out.append({
            col: g, "n": len(dfg),
            "pos_rate": dfg["y"].mean(),
            "pr_auc": prg, "roc_auc": rocg, "brier": bg,
            "tpr": repg["tpr"], "precision": repg["precision"]
        })
    return pd.DataFrame(out).sort_values("pr_auc", ascending=False)

if "gender" in df_err.columns:
    print("\nGroup report: gender")
    display(group_report(df_err, "gender"))

if "Race" in df_err.columns:
    print("\nGroup report: Race")
    display(group_report(df_err, "Race"))

[fold 1 ep 00] loss=0.6637 PR=0.1222 ROC=0.6404 Brier=0.5329
[fold 1 ep 01] loss=0.4853 PR=0.1521 ROC=0.6761 Brier=0.4807
[fold 1 ep 02] loss=0.4756 PR=0.1572 ROC=0.6862 Brier=0.4939
[fold 1 ep 03] loss=0.4432 PR=0.1634 ROC=0.6906 Brier=0.4626
[fold 1 ep 04] loss=0.4474 PR=0.1802 ROC=0.7066 Brier=0.4171
[fold 1 ep 05] loss=0.4306 PR=0.1890 ROC=0.7042 Brier=0.4580
[fold 1 ep 06] loss=0.4240 PR=0.1942 ROC=0.7020 Brier=0.4403
[fold 1 ep 07] loss=0.4145 PR=0.1989 ROC=0.7052 Brier=0.4152
[fold 1 ep 08] loss=0.4060 PR=0.1923 ROC=0.7062 Brier=0.4073
[fold 1 ep 09] loss=0.3908 PR=0.1999 ROC=0.7088 Brier=0.3786
[fold 1 ep 10] loss=0.3703 PR=0.1800 ROC=0.7087 Brier=0.4343
[fold 1 ep 11] loss=0.3733 PR=0.1720 ROC=0.7064 Brier=0.3295
[fold 1 ep 12] loss=0.3515 PR=0.1683 ROC=0.7034 Brier=0.3502
[fold 1 ep 13] loss=0.3305 PR=0.1662 ROC=0.6992 Brier=0.3225
[fold 1 ep 14] loss=0.3203 PR=0.1578 ROC=0.6990 Brier=0.3540
[fold 1 ep 15] loss=0.3283 PR=0.1604 ROC=0.6985 Brier=0.3095
[fold 1 ep 16] loss=0.30

,gender,age,Race,Marital status,alcohol,smoke,sleep disorder,Health Insurance,General health condition,depression,...,Dietary fiber,Total fat,Total saturated fatty acids,Total monounsaturated fatty acids,Total polyunsaturated fatty acids,Potassium,Sodium,y,p,pred
1443,1,1,1,3,1,0,2,2,3,1,...,13.3,57.67,18.563,25.823,7.273,3645,3096,1,0.022588,0
574,1,2,3,5,0,0,2,1,4,1,...,40.4,149.44,41.908,63.372,30.079,4871,7600,1,0.023917,0
1409,2,2,4,4,1,0,2,1,3,1,...,30.2,38.14,15.693,10.257,6.919,4552,2838,1,0.029410,0
622,1,2,3,5,1,1,1,1,3,1,...,8.7,68.17,25.184,26.180,10.073,868,2480,1,0.029731,0
3546,2,1,4,3,1,1,1,2,4,2,...,12.6,67.57,24.605,22.668,15.083,2112,3070,1,0.035585,0
594,2,2,3,6,1,1,1,1,3,2,...,2.8,39.55,8.358,8.938,16.383,1457,3069,1,0.036950,0
4264,2,1,4,4,1,0,2,1,3,1,...,21.7,112.06,36.803,35.538,33.056,1813,3628,1,0.037944,0
4408,1,3,5,3,1,1,2,1,3,1,...,14.4,155.26,52.552,49.801,40.359,3438,4994,1,0.038962,0
441,1,2,3,5,1,1,2,1,2,1,...,14.5,99.21,25.848,37.079,23.533,3468,8196,1,0.040654,0
376,1,2,3,1,1,0,1,2,3,1,...,9.2,85.35,28.092,32.591,16.816,2473,5343,1,0.047777,0



Top 10 highest-confidence FALSE POSITIVES:


,gender,age,Race,Marital status,alcohol,smoke,sleep disorder,Health Insurance,General health condition,depression,...,Dietary fiber,Total fat,Total saturated fatty acids,Total monounsaturated fatty acids,Total polyunsaturated fatty acids,Potassium,Sodium,y,p,pred
4313,1,3,3,2,1,1,2,1,4,1,...,15.8,178.81,61.648,60.258,41.707,3025,3852,0,0.861905,1
533,2,3,3,1,0,1,2,1,4,3,...,2.7,19.13,8.089,6.157,2.137,714,615,0,0.858920,1
4327,1,2,3,5,1,1,1,1,4,3,...,8.1,50.29,22.562,15.369,7.606,1688,1635,0,0.858735,1
1089,2,2,3,1,1,1,2,1,4,2,...,15.4,136.87,59.821,41.871,25.444,2965,5434,0,0.858706,1
2702,1,3,3,1,1,1,2,1,4,1,...,10.9,67.44,15.441,22.304,25.725,1480,2658,0,0.858619,1
4279,1,3,3,1,1,1,2,1,4,1,...,4.8,52.00,18.043,16.325,12.393,1510,2064,0,0.858325,1
411,2,3,3,1,0,1,2,1,4,1,...,9.6,63.94,15.972,23.198,20.685,1003,2242,0,0.856920,1
4183,2,3,3,2,1,1,1,1,4,1,...,8.5,103.37,39.220,32.256,23.433,1129,2382,0,0.856588,1
3330,2,3,3,2,1,0,2,1,4,2,...,18.2,150.48,42.992,60.014,33.009,2498,6103,0,0.855658,1
4390,2,3,3,1,1,0,2,1,4,1,...,8.5,96.08,31.852,29.757,25.478,1123,2575,0,0.855627,1



Group report: gender


,gender,n,pos_rate,pr_auc,roc_auc,brier,tpr,precision
0,1,2093,0.077401,0.154534,0.703816,0.310191,0.685185,0.149194
1,2,2510,0.079681,0.138616,0.670597,0.337868,0.400000,0.161616



Group report: Race


,Race,n,pos_rate,pr_auc,roc_auc,brier,tpr,precision
3,4,1101,0.086285,0.163850,0.678100,0.334515,0.473684,0.161871
2,3,2192,0.089416,0.156836,0.684065,0.353926,0.510204,0.165017
0,1,518,0.059846,0.135238,0.699742,0.318396,0.548387,0.137097
4,5,345,0.052174,0.123027,0.682467,0.213440,0.388889,0.194444
1,2,447,0.049217,0.060964,0.594973,0.256390,0.818182,0.073469


In [7]:
# =========================
# CELL 11 (REPLACE): FOLD-WISE DL PREPROCESS
# =========================
from sklearn.preprocessing import StandardScaler

def make_fold_tensors(X_df, y, tr_idx, va_idx, cat_cols, num_cols):
    X_tr = X_df.iloc[tr_idx].copy()
    X_va = X_df.iloc[va_idx].copy()
    y_tr = y[tr_idx].astype(np.float32)
    y_va = y[va_idx].astype(np.float32)

    # ---- categorical mapping fitted on TRAIN only
    Xc_tr_list, Xc_va_list = [], []
    cat_dims = []
    for c in cat_cols:
        tr_cat = pd.Series(X_tr[c].astype(int).values)
        va_cat = pd.Series(X_va[c].astype(int).values)

        # build mapping from train unique values
        uniq = tr_cat.unique().tolist()
        mapping = {v: i+1 for i, v in enumerate(sorted(uniq))}  # 0 reserved for UNK

        Xc_tr = tr_cat.map(mapping).fillna(0).astype(int).values
        Xc_va = va_cat.map(mapping).fillna(0).astype(int).values

        Xc_tr_list.append(Xc_tr)
        Xc_va_list.append(Xc_va)

        cat_dims.append(len(mapping) + 1)  # +1 because 0 exists

    X_cat_tr = np.stack(Xc_tr_list, axis=1).astype(np.int64)
    X_cat_va = np.stack(Xc_va_list, axis=1).astype(np.int64)

    # ---- numeric scaler fitted on TRAIN only
    scaler = StandardScaler()
    X_num_tr = scaler.fit_transform(X_tr[num_cols].values.astype(np.float32))
    X_num_va = scaler.transform(X_va[num_cols].values.astype(np.float32))

    return (X_cat_tr, X_num_tr, y_tr), (X_cat_va, X_num_va, y_va), cat_dims

# =========================
# CELL 15 (REPLACE): DATALOADERS (NO SAMPLER)
# =========================
from torch.utils.data import DataLoader

def make_loaders_simple(tr_pack, va_pack, batch_size=256):
    Xc_tr, Xn_tr, y_tr = tr_pack
    Xc_va, Xn_va, y_va = va_pack

    ds_tr = TabularDataset(Xc_tr, Xn_tr, y_tr)
    ds_va = TabularDataset(Xc_va, Xn_va, y_va)

    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, drop_last=False)
    dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False, drop_last=False)
    return dl_tr, dl_va

# =========================
# CELL 16 (REPLACE): TRAIN ONE FOLD (BEST EPOCH PRED)
# =========================
def train_one_fold_fixed(
    X_df, y, tr_idx, va_idx, fold,
    cat_cols, num_cols,
    d_model=96, n_heads=4, n_layers=3, dropout=0.15,
    lr=3e-4, weight_decay=1e-4,
    epochs=60, patience=10, batch_size=256,
    focal_gamma=1.5
):
    tr_pack, va_pack, cat_dims = make_fold_tensors(X_df, y, tr_idx, va_idx, cat_cols, num_cols)
    dl_tr, dl_va = make_loaders_simple(tr_pack, va_pack, batch_size=batch_size)

    # pos_weight computed on TRAIN only
    y_tr = tr_pack[2]
    pos = float((y_tr == 1).sum())
    neg = float((y_tr == 0).sum())
    pos_w = neg / max(pos, 1.0)

    model = TabularTransformer(
        cat_dims=cat_dims, num_dim=len(num_cols),
        d_model=d_model, n_heads=n_heads, n_layers=n_layers, dropout=dropout
    ).to(DEVICE)

    pos_weight = torch.tensor([pos_w], dtype=torch.float32, device=DEVICE)
    criterion = FocalBCE(pos_weight=pos_weight, gamma=focal_gamma).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_pr = -1.0
    best_state = None
    bad = 0
    history = []

    for ep in range(epochs):
        model.train()
        tr_losses = []
        for xb_cat, xb_num, yb in dl_tr:
            xb_cat, xb_num, yb = xb_cat.to(DEVICE), xb_num.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(xb_cat, xb_num)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_losses.append(loss.item())

        # ---- validate
        model.eval()
        p_va_all, y_va_all = [], []
        with torch.no_grad():
            for xb_cat, xb_num, yb in dl_va:
                xb_cat, xb_num = xb_cat.to(DEVICE), xb_num.to(DEVICE)
                logits = model(xb_cat, xb_num)
                p = torch.sigmoid(logits)
                p_va_all.append(to_np(p))
                y_va_all.append(to_np(yb))

        p_va = np.concatenate(p_va_all)
        y_va = np.concatenate(y_va_all).astype(int)

        pr, roc, brier = eval_binary_full(y_va, p_va)
        tr_loss = float(np.mean(tr_losses))
        history.append({"fold": fold, "epoch": ep, "tr_loss": tr_loss, "pr": pr, "roc": roc, "brier": brier})
        print(f"[fold {fold} ep {ep:02d}] loss={tr_loss:.4f} PR={pr:.4f} ROC={roc:.4f} Brier={brier:.4f}")

        # early stop on PR
        if pr > best_pr + 1e-4:
            best_pr = pr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    # ---- load best model
    model.load_state_dict(best_state)

    # ---- recompute p_va using BEST model (bug fix)
    model.eval()
    p_va_all, y_va_all = [], []
    with torch.no_grad():
        for xb_cat, xb_num, yb in dl_va:
            xb_cat, xb_num = xb_cat.to(DEVICE), xb_num.to(DEVICE)
            logits = model(xb_cat, xb_num)
            p = torch.sigmoid(logits)
            p_va_all.append(to_np(p))
            y_va_all.append(to_np(yb))

    p_va_best = np.concatenate(p_va_all)
    y_va_best = np.concatenate(y_va_all).astype(int)

    return model, history, (y_va_best, p_va_best)

# =========================
# CELL 17 (REPLACE): CV TRAIN (FIXED)
# =========================
set_seed(cfg.seed)

all_hist = []
oof = np.zeros(len(y), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_, y), 1):
    model, hist, (y_va, p_va) = train_one_fold_fixed(
        X_df=X_,
        y=y,
        tr_idx=tr_idx, va_idx=va_idx,
        fold=fold,
        cat_cols=cat_cols, num_cols=num_cols,
        d_model=96, n_heads=4, n_layers=3, dropout=0.15,
        lr=3e-4, weight_decay=1e-4,
        epochs=60, patience=10,
        batch_size=256,
        focal_gamma=1.5
    )
    all_hist.extend(hist)
    oof[va_idx] = p_va

pr, roc, brier = eval_binary_full(y, oof)
print("\n==== OOF OVERALL (FIXED) ====")
print("PR-AUC:", pr)
print("ROC-AUC:", roc)
print("Brier:", brier)

hist_df = pd.DataFrame(all_hist)
hist_df.to_csv(os.path.join(TAB_DIR, "ft_transformer_cv_history_fixed.csv"), index=False)

[fold 1 ep 00] loss=0.4512 PR=0.1375 ROC=0.6608 Brier=0.2185
[fold 1 ep 01] loss=0.4335 PR=0.1465 ROC=0.6688 Brier=0.2361
[fold 1 ep 02] loss=0.4169 PR=0.1647 ROC=0.6853 Brier=0.2552
[fold 1 ep 03] loss=0.4112 PR=0.1718 ROC=0.6910 Brier=0.2265
[fold 1 ep 04] loss=0.4056 PR=0.1704 ROC=0.6939 Brier=0.2392
[fold 1 ep 05] loss=0.4035 PR=0.1702 ROC=0.6968 Brier=0.2431
[fold 1 ep 06] loss=0.3931 PR=0.1712 ROC=0.6994 Brier=0.2224
[fold 1 ep 07] loss=0.3933 PR=0.1761 ROC=0.6984 Brier=0.2334
[fold 1 ep 08] loss=0.3967 PR=0.1746 ROC=0.7025 Brier=0.2471
[fold 1 ep 09] loss=0.3911 PR=0.1735 ROC=0.7021 Brier=0.2327
[fold 1 ep 10] loss=0.3836 PR=0.1837 ROC=0.7036 Brier=0.1990
[fold 1 ep 11] loss=0.3788 PR=0.1760 ROC=0.6998 Brier=0.1989
[fold 1 ep 12] loss=0.3716 PR=0.1694 ROC=0.7037 Brier=0.1834
[fold 1 ep 13] loss=0.3658 PR=0.1750 ROC=0.7007 Brier=0.2081
[fold 1 ep 14] loss=0.3644 PR=0.1618 ROC=0.7072 Brier=0.2098
[fold 1 ep 15] loss=0.3611 PR=0.1619 ROC=0.7031 Brier=0.2007
[fold 1 ep 16] loss=0.35

In [8]:
# =========================
# CELL 19: TRAIN ONE FOLD WITH SEED (FOR ENSEMBLE)
# =========================
def train_one_fold_seeded(seed, X_df, y, tr_idx, va_idx, fold, cat_cols, num_cols, **kwargs):
    set_seed(seed)
    model, hist, (y_va, p_va) = train_one_fold_fixed(
        X_df=X_df, y=y, tr_idx=tr_idx, va_idx=va_idx, fold=fold,
        cat_cols=cat_cols, num_cols=num_cols,
        **kwargs
    )
    return model, hist, (y_va, p_va)

# =========================
# CELL 20: ENSEMBLE OOF (M=5)
# =========================
ENSEMBLE_SEEDS = [0, 1, 2, 3, 4]   # M=5
oof_ens = np.zeros(len(y), dtype=np.float32)

# store fold-wise predictions for analysis if needed
fold_preds = {}

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_, y), 1):
    preds = []
    for s in ENSEMBLE_SEEDS:
        _, hist, (y_va, p_va) = train_one_fold_seeded(
            seed=s,
            X_df=X_, y=y,
            tr_idx=tr_idx, va_idx=va_idx,
            fold=fold,
            cat_cols=cat_cols, num_cols=num_cols,
            # keep your current best settings
            d_model=96, n_heads=4, n_layers=3, dropout=0.15,
            lr=3e-4, weight_decay=1e-4,
            epochs=60, patience=10, batch_size=256,
            focal_gamma=1.5
        )
        preds.append(p_va)

    p_va_ens = np.mean(np.stack(preds, axis=0), axis=0)
    oof_ens[va_idx] = p_va_ens
    fold_preds[fold] = (va_idx, p_va_ens)

pr, roc, brier = eval_binary_full(y, oof_ens)
print("\n==== OOF OVERALL (ENSEMBLE) ====")
print("PR-AUC:", pr)
print("ROC-AUC:", roc)
print("Brier:", brier)

[fold 1 ep 00] loss=0.4499 PR=0.1659 ROC=0.6816 Brier=0.2268
[fold 1 ep 01] loss=0.4211 PR=0.1568 ROC=0.6768 Brier=0.2375
[fold 1 ep 02] loss=0.4143 PR=0.1701 ROC=0.6853 Brier=0.2358
[fold 1 ep 03] loss=0.4005 PR=0.1789 ROC=0.6940 Brier=0.1991
[fold 1 ep 04] loss=0.3967 PR=0.1722 ROC=0.6935 Brier=0.2073
[fold 1 ep 05] loss=0.4026 PR=0.1762 ROC=0.6959 Brier=0.2197
[fold 1 ep 06] loss=0.4002 PR=0.1743 ROC=0.6957 Brier=0.2170
[fold 1 ep 07] loss=0.3925 PR=0.1829 ROC=0.7032 Brier=0.2630
[fold 1 ep 08] loss=0.3909 PR=0.1708 ROC=0.7027 Brier=0.2301
[fold 1 ep 09] loss=0.3836 PR=0.1689 ROC=0.7024 Brier=0.2096
[fold 1 ep 10] loss=0.3767 PR=0.1600 ROC=0.7018 Brier=0.2282
[fold 1 ep 11] loss=0.3780 PR=0.1685 ROC=0.7037 Brier=0.1963
[fold 1 ep 12] loss=0.3697 PR=0.1735 ROC=0.7098 Brier=0.2067
[fold 1 ep 13] loss=0.3693 PR=0.1777 ROC=0.7088 Brier=0.1917
[fold 1 ep 14] loss=0.3642 PR=0.1478 ROC=0.7038 Brier=0.1743
[fold 1 ep 15] loss=0.3520 PR=0.1622 ROC=0.6985 Brier=0.2039
[fold 1 ep 16] loss=0.34

In [9]:
# =========================
# CELL 21: ERROR PROFILING (WHY MISTAKES)
# =========================
best_thr = best_threshold_by_f1(y, oof_ens)
rep = threshold_report(y, oof_ens, best_thr)
print("Best-F1 threshold:", best_thr)
print(rep)

df_err = X_.copy()
df_err["y"] = y
df_err["p"] = oof_ens
df_err["pred"] = (oof_ens >= best_thr).astype(int)

fn_df = df_err[(df_err["y"]==1) & (df_err["pred"]==0)].copy()
tp_df = df_err[(df_err["y"]==1) & (df_err["pred"]==1)].copy()

print("\nFN count:", len(fn_df), "TP count:", len(tp_df))

# Compare FN vs TP on key clinical risk factors
key_feats = [
    "age", "hypertension", "diabetes", "high cholesterol",
    "Systolic blood pressure", "Diastolic blood pressure",
    "Body Mass Index", "sleep time", "Minutes sedentary activity"
]
key_feats = [c for c in key_feats if c in df_err.columns]

summary = []
for c in key_feats:
    summary.append({
        "feature": c,
        "TP_mean": tp_df[c].mean(),
        "FN_mean": fn_df[c].mean(),
        "TP_median": tp_df[c].median(),
        "FN_median": fn_df[c].median(),
    })
display(pd.DataFrame(summary))

print("\nTop 20 hardest FN (lowest predicted risk among true strokes):")
display(fn_df.sort_values("p").head(20))

Best-F1 threshold: 0.5911094
{'thr': np.float32(0.5911094), 'acc': np.float64(0.8674777319139692), 'bal_acc': np.float64(0.6084633562656566), 'precision': np.float64(0.2339055793991411), 'tpr': np.float64(0.3011049723756898), 'fpr': np.float64(0.08417825984437631), 'tp': np.int64(109), 'fp': np.int64(357), 'tn': np.int64(3884), 'fn': np.int64(253)}

FN count: 253 TP count: 109


,feature,TP_mean,FN_mean,TP_median,FN_median
0,age,2.715596,2.482213,3.0,3.0
1,hypertension,0.963303,0.869565,1.0,1.0
2,diabetes,0.458716,0.300395,0.0,0.0
3,high cholesterol,0.798165,0.624506,1.0,1.0
4,Systolic blood pressure,138.183486,135.185771,136.0,134.0
5,Diastolic blood pressure,66.385321,71.193676,64.0,72.0
6,Body Mass Index,3.110092,3.320158,3.0,4.0
7,sleep time,6.889908,7.077075,7.0,7.0
8,Minutes sedentary activity,456.605505,367.648221,480.0,360.0



Top 20 hardest FN (lowest predicted risk among true strokes):


,gender,age,Race,Marital status,alcohol,smoke,sleep disorder,Health Insurance,General health condition,depression,...,Dietary fiber,Total fat,Total saturated fatty acids,Total monounsaturated fatty acids,Total polyunsaturated fatty acids,Potassium,Sodium,y,p,pred
1443,1,1,1,3,1,0,2,2,3,1,...,13.3,57.67,18.563,25.823,7.273,3645,3096,1,0.176860,0
376,1,2,3,1,1,0,1,2,3,1,...,9.2,85.35,28.092,32.591,16.816,2473,5343,1,0.246739,0
4024,1,2,2,1,1,0,1,1,2,1,...,6.0,99.22,33.696,40.289,16.912,1694,4561,1,0.253239,0
3421,2,1,4,5,0,0,1,1,2,1,...,22.0,103.25,40.588,37.010,14.773,2054,3996,1,0.260549,0
3991,2,1,1,3,1,1,1,2,4,1,...,24.7,154.31,42.074,51.793,50.443,3343,6062,1,0.261275,0
1377,1,3,2,1,1,1,2,1,2,1,...,13.8,99.52,36.260,32.573,19.457,3397,3670,1,0.271948,0
2164,2,2,4,1,1,0,2,1,3,1,...,9.9,49.31,9.503,14.888,22.161,1581,1636,1,0.306225,0
2432,1,2,1,4,1,1,2,2,3,1,...,29.0,86.01,23.366,30.852,22.545,2617,2845,1,0.306367,0
1216,1,3,3,1,1,0,2,1,2,1,...,27.3,81.30,18.314,28.462,28.370,3088,3943,1,0.309544,0
441,1,2,3,5,1,1,2,1,2,1,...,14.5,99.21,25.848,37.079,23.533,3468,8196,1,0.313573,0


In [10]:
# =========================
# CELL 22: SCREENING OPERATING POINTS
# =========================
def recall_at_precision(y_true, p, min_precision=0.20):
    prec, rec, thr = precision_recall_curve(y_true, p)
    # thr has length-1 relative to prec/rec
    valid = np.where(prec[:-1] >= min_precision)[0]
    if len(valid) == 0:
        return None
    i = valid[np.argmax(rec[valid])]
    return {"min_precision": min_precision, "threshold": thr[i], "precision": prec[i], "recall": rec[i]}

def capture_at_topk(y_true, p, top_frac=0.10):
    k = int(len(p) * top_frac)
    idx = np.argsort(-p)[:k]
    return {"top_frac": top_frac, "k": k, "capture_rate": y_true[idx].mean(), "pos_captured": int(y_true[idx].sum())}

print("Recall@Precision≥0.20:", recall_at_precision(y, oof_ens, 0.20))
print("Recall@Precision≥0.25:", recall_at_precision(y, oof_ens, 0.25))
print("Top 5% capture:", capture_at_topk(y, oof_ens, 0.05))
print("Top 10% capture:", capture_at_topk(y, oof_ens, 0.10))

Recall@Precision≥0.20: {'min_precision': 0.2, 'threshold': np.float32(0.5799352), 'precision': np.float64(0.2), 'recall': np.float64(0.3314917127071823)}
Recall@Precision≥0.25: {'min_precision': 0.25, 'threshold': np.float32(0.6124709), 'precision': np.float64(0.25), 'recall': np.float64(0.18232044198895028)}
Top 5% capture: {'top_frac': 0.05, 'k': 230, 'capture_rate': np.float64(0.24782608695652175), 'pos_captured': 57}
Top 10% capture: {'top_frac': 0.1, 'k': 460, 'capture_rate': np.float64(0.23043478260869565), 'pos_captured': 106}


In [13]:
# =========================
# CELL 23: ECE + RELIABILITY DIAGRAM
# =========================
def expected_calibration_error(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = p[mask].mean()
        w = mask.mean()
        ece += w * abs(acc - conf)
    return float(ece)

def reliability_data(y_true, p, n_bins=15):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins+1)
    out = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (p >= lo) & (p < hi) if i < n_bins-1 else (p >= lo) & (p <= hi)
        if mask.sum() == 0:
            out.append((0.5*(lo+hi), np.nan, np.nan, 0))
            continue
        acc = y_true[mask].mean()
        conf = p[mask].mean()
        out.append((0.5*(lo+hi), acc, conf, int(mask.sum())))
    return out

def plot_reliability(y_true, p, title, path, n_bins=15):
    rd = reliability_data(y_true, p, n_bins=n_bins)
    xs = [r[0] for r in rd]
    accs = [r[1] for r in rd]
    confs = [r[2] for r in rd]

    plt.figure()
    plt.plot([0,1],[0,1])
    plt.plot(xs, confs, marker='o', label="Mean predicted")
    plt.plot(xs, accs, marker='o', label="Empirical accuracy")
    plt.xlabel("Confidence bin center")
    plt.ylabel("Value")
    plt.title(title)
    plt.legend()
    save_fig(path)

# =========================
# CELL 24: TEMPERATURE SCALING ON OOF (LOGIT-SPACE)
# =========================
def logit(p):
    p = np.clip(p, 1e-12, 1-1e-12)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

def fit_temperature(y_true, p, max_iter=2000, lr=0.01):
    """
    Fit T to minimize NLL on: sigmoid(logit(p)/T)
    using simple gradient descent.
    """
    y_true = np.asarray(y_true).astype(np.float32)
    z = logit(p).astype(np.float32)

    T = 1.0
    for _ in range(max_iter):
        # forward
        zT = z / T
        q = sigmoid(zT)
        q = np.clip(q, 1e-12, 1-1e-12)

        # NLL gradient wrt T
        # d/dT z/T = -z/T^2
        dqdzt = q*(1-q)
        dzt_dT = -z/(T*T)
        # dNLL/dq = -(y/q) + (1-y)/(1-q)
        dNLL_dq = -(y_true/q) + (1-y_true)/(1-q)
        dNLL_dT = np.mean(dNLL_dq * dqdzt * dzt_dT)

        # update
        T_new = T - lr * dNLL_dT
        T_new = float(np.clip(T_new, 0.05, 10.0))
        if abs(T_new - T) < 1e-8:
            break
        T = T_new
    return float(T)

def apply_temperature(p, T):
    z = logit(p)
    return sigmoid(z / T)

# Fit T on OOF
T_hat = fit_temperature(y, oof_ens, max_iter=4000, lr=0.05)
oof_ts = apply_temperature(oof_ens, T_hat)

print("Estimated T:", T_hat)

# Metrics before/after
pr0, roc0, br0 = eval_binary_full(y, oof_ens)
pr1, roc1, br1 = eval_binary_full(y, oof_ts)
ece0 = expected_calibration_error(y, oof_ens, n_bins=15)
ece1 = expected_calibration_error(y, oof_ts, n_bins=15)

print("\n=== BEFORE TS ===")
print("PR:", pr0, "ROC:", roc0, "Brier:", br0, "ECE:", ece0)

print("\n=== AFTER TS ===")
print("PR:", pr1, "ROC:", roc1, "Brier:", br1, "ECE:", ece1)

# =========================
# CELL 25: RELIABILITY PLOTS
# =========================
plot_reliability(y, oof_ens, "Reliability (OOF Ensemble, Uncalibrated)",
                 os.path.join(FIG_DIR, "reliability_uncalibrated.png"), n_bins=15)
plot_reliability(y, oof_ts, f"Reliability (OOF Ensemble, TempScaled T={T_hat:.3f})",
                 os.path.join(FIG_DIR, "reliability_tempscaled.png"), n_bins=15)

print("Saved reliability figures to:", FIG_DIR)

# =========================
# CELL 26: MEMBER-WISE OOF FOR UNCERTAINTY
# =========================
ENSEMBLE_SEEDS = [0, 1, 2, 3, 4]
P_members = np.zeros((len(ENSEMBLE_SEEDS), len(y)), dtype=np.float32)

for mi, s in enumerate(ENSEMBLE_SEEDS):
    oof_m = np.zeros(len(y), dtype=np.float32)
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_, y), 1):
        _, _, (y_va, p_va) = train_one_fold_seeded(
            seed=s, X_df=X_, y=y,
            tr_idx=tr_idx, va_idx=va_idx,
            fold=fold, cat_cols=cat_cols, num_cols=num_cols,
            d_model=96, n_heads=4, n_layers=3, dropout=0.15,
            lr=3e-4, weight_decay=1e-4,
            epochs=60, patience=10, batch_size=256,
            focal_gamma=1.5
        )
        oof_m[va_idx] = p_va
    P_members[mi] = oof_m
    pr_m, roc_m, br_m = eval_binary_full(y, oof_m)
    print(f"[member {mi} seed {s}] PR={pr_m:.4f} ROC={roc_m:.4f} Brier={br_m:.4f}")

# ensemble mean and uncertainty
p_mean = P_members.mean(axis=0)
p_std  = P_members.std(axis=0)   # epistemic proxy

print("\nSanity: ensemble mean matches previous?")
print("mean abs diff:", float(np.mean(np.abs(p_mean - oof_ens))))

# =========================
# CELL 27: RISK–COVERAGE (SELECTIVE PREDICTION)
# =========================
def risk_coverage(y_true, p, u, cover_fracs=np.linspace(0.2, 1.0, 9)):
    """
    Sort by uncertainty u ascending (most confident first).
    Evaluate PR-AUC at different coverage levels.
    """
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    u = np.asarray(u).astype(float)

    order = np.argsort(u)  # low uncertainty first
    out = []
    for cf in cover_fracs:
        k = int(len(y_true) * cf)
        idx = order[:k]
        pr = average_precision_score(y_true[idx], p[idx]) if y_true[idx].sum() > 0 else np.nan
        roc = roc_auc_score(y_true[idx], p[idx]) if len(np.unique(y_true[idx])) > 1 else np.nan
        out.append({"coverage": cf, "n": k, "pos": int(y_true[idx].sum()), "pr_auc": pr, "roc_auc": roc})
    return pd.DataFrame(out)

rc = risk_coverage(y, p_mean, p_std, cover_fracs=np.linspace(0.2, 1.0, 9))
display(rc)

plt.figure()
plt.plot(rc["coverage"], rc["pr_auc"], marker="o")
plt.xlabel("Coverage (fraction kept)")
plt.ylabel("PR-AUC")
plt.title("Risk–Coverage Curve (lower uncertainty → better PR)")
save_fig(os.path.join(FIG_DIR, "risk_coverage_pr.png"))
print("Saved:", os.path.join(FIG_DIR, "risk_coverage_pr.png"))

Estimated T: 0.4458076059818268

=== BEFORE TS ===
PR: 0.17481735655754735 ROC: 0.712555414716377 Brier: 0.2126400060345197 ECE: 0.37450695341174145

=== AFTER TS ===
PR: 0.17481735655754735 ROC: 0.712555414716377 Brier: 0.20315669465104616 ECE: 0.3336879484170925
Saved reliability figures to: ./outputs/figures
[fold 1 ep 00] loss=0.4499 PR=0.1659 ROC=0.6816 Brier=0.2268
[fold 1 ep 01] loss=0.4211 PR=0.1568 ROC=0.6768 Brier=0.2375
[fold 1 ep 02] loss=0.4143 PR=0.1701 ROC=0.6853 Brier=0.2358
[fold 1 ep 03] loss=0.4005 PR=0.1789 ROC=0.6940 Brier=0.1991
[fold 1 ep 04] loss=0.3967 PR=0.1722 ROC=0.6935 Brier=0.2073
[fold 1 ep 05] loss=0.4026 PR=0.1762 ROC=0.6959 Brier=0.2197
[fold 1 ep 06] loss=0.4002 PR=0.1743 ROC=0.6957 Brier=0.2170
[fold 1 ep 07] loss=0.3925 PR=0.1829 ROC=0.7032 Brier=0.2630
[fold 1 ep 08] loss=0.3909 PR=0.1708 ROC=0.7027 Brier=0.2301
[fold 1 ep 09] loss=0.3836 PR=0.1689 ROC=0.7024 Brier=0.2096
[fold 1 ep 10] loss=0.3767 PR=0.1600 ROC=0.7018 Brier=0.2282
[fold 1 ep 11] l

,coverage,n,pos,pr_auc,roc_auc
0,0.2,920,105,0.232162,0.752112
1,0.3,1380,142,0.221440,0.749824
2,0.4,1841,179,0.208785,0.725679
3,0.5,2301,214,0.198481,0.716156
4,0.6,2761,248,0.191600,0.709523
5,0.7,3222,285,0.186735,0.702924
6,0.8,3682,312,0.181451,0.704100
7,0.9,4142,339,0.178556,0.706778
8,1.0,4603,362,0.174817,0.712555


Saved: ./outputs/figures/risk_coverage_pr.png


In [14]:
# =========================
# CELL 28: FAIRNESS METRICS
# =========================

def group_metrics(df, y_true, p, group_col):
    out = []
    for g, idx in df.groupby(group_col).groups.items():
        idx = np.array(list(idx))
        if len(idx) < 50:
            continue
        
        yg = y_true[idx]
        pg = p[idx]
        
        pr, roc, br = eval_binary_full(yg, pg)
        ece = expected_calibration_error(yg, pg, n_bins=15)
        
        thr = best_threshold_by_f1(yg, pg)
        rep = threshold_report(yg, pg, thr)
        
        out.append({
            group_col: g,
            "n": len(idx),
            "pos_rate": yg.mean(),
            "pr_auc": pr,
            "roc_auc": roc,
            "brier": br,
            "ece": ece,
            "tpr": rep["tpr"],
            "fnr": 1 - rep["tpr"],
            "precision": rep["precision"]
        })
    return pd.DataFrame(out).sort_values("pr_auc", ascending=False)

df_base = X_.copy()
print("=== Gender ===")
display(group_metrics(df_base, y, oof_ts, "gender"))

print("=== Race ===")
display(group_metrics(df_base, y, oof_ts, "Race"))

print("=== Health Insurance ===")
display(group_metrics(df_base, y, oof_ts, "Health Insurance"))

# =========================
# CELL 29: FAIRNESS GAP SUMMARY
# =========================

def fairness_gap(dfm, metric):
    return float(dfm[metric].max() - dfm[metric].min())

g_gender = group_metrics(df_base, y, oof_ts, "gender")
g_race   = group_metrics(df_base, y, oof_ts, "Race")

print("Gender gaps:")
print("PR gap:", fairness_gap(g_gender, "pr_auc"))
print("TPR gap:", fairness_gap(g_gender, "tpr"))
print("ECE gap:", fairness_gap(g_gender, "ece"))

print("\nRace gaps:")
print("PR gap:", fairness_gap(g_race, "pr_auc"))
print("TPR gap:", fairness_gap(g_race, "tpr"))
print("ECE gap:", fairness_gap(g_race, "ece"))

# =========================
# CELL 30: GROUP-WISE THRESHOLDING (FAIRNESS MITIGATION)
# =========================

def group_threshold_predictions(df, y_true, p, group_col):
    preds = np.zeros_like(p, dtype=int)
    thresholds = {}
    
    for g, idx in df.groupby(group_col).groups.items():
        idx = np.array(list(idx))
        if len(idx) < 50:
            continue
        
        yg = y_true[idx]
        pg = p[idx]
        
        thr = best_threshold_by_f1(yg, pg)
        thresholds[g] = thr
        preds[idx] = (pg >= thr).astype(int)
    
    return preds, thresholds

# Apply on gender
pred_gender, thr_gender = group_threshold_predictions(df_base, y, oof_ts, "gender")

tn, fp, fn, tp = confusion_matrix(y, pred_gender).ravel()
print("Confusion after gender-thresholding:")
print("TP:", tp, "FP:", fp, "TN:", tn, "FN:", fn)

print("Group thresholds:", thr_gender)

=== Gender ===


,gender,n,pos_rate,pr_auc,roc_auc,brier,ece,tpr,fnr,precision
0,1,2093,0.077401,0.180989,0.714521,0.198126,0.324276,0.296296,0.703704,0.222222
1,2,2510,0.079681,0.175375,0.710716,0.207352,0.341537,0.305000,0.695000,0.244980


=== Race ===


,Race,n,pos_rate,pr_auc,roc_auc,brier,ece,tpr,fnr,precision
3,4,1101,0.086285,0.206108,0.713812,0.212842,0.345702,0.284211,0.715789,0.275510
2,3,2192,0.089416,0.189395,0.712928,0.225032,0.355272,0.377551,0.622449,0.244224
0,1,518,0.059846,0.154221,0.668742,0.177415,0.310943,0.645161,0.354839,0.129032
4,5,345,0.052174,0.144640,0.751104,0.129962,0.255403,0.555556,0.444444,0.169492
1,2,447,0.049217,0.076948,0.622781,0.158353,0.287035,0.500000,0.500000,0.090909


=== Health Insurance ===


,Health Insurance,n,pos_rate,pr_auc,roc_auc,brier,ece,tpr,fnr,precision
0,1,4010,0.083791,0.181789,0.711556,0.214096,0.344950,0.324405,0.675595,0.236443
1,2,593,0.043845,0.089043,0.628612,0.129182,0.257627,0.346154,0.653846,0.116883


Gender gaps:
PR gap: 0.00561449062767172
TPR gap: 0.008703703703703991
ECE gap: 0.0172608737236562

Race gaps:
PR gap: 0.1291599823225129
TPR gap: 0.3609507640067734
ECE gap: 0.09986888151014389
Confusion after gender-thresholding:
TP: 109 FP: 356 TN: 3885 FN: 253
Group thresholds: {1: np.float32(0.6970487), 2: np.float32(0.69565874)}


In [15]:
# =========================
# CELL E1: FULL METRICS SUITE
# =========================
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    matthews_corrcoef, cohen_kappa_score,
    log_loss, confusion_matrix
)

p_use = oof_ts  # calibrated; change to oof_ens if you prefer uncalibrated

def compute_metrics_at_threshold(y_true, p, thr):
    y_hat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()

    out = {}
    out["threshold"] = float(thr)
    out["TP"] = int(tp); out["FP"] = int(fp); out["TN"] = int(tn); out["FN"] = int(fn)

    out["accuracy"] = accuracy_score(y_true, y_hat)
    out["balanced_accuracy"] = balanced_accuracy_score(y_true, y_hat)
    out["precision"] = precision_score(y_true, y_hat, zero_division=0)
    out["recall_tpr"] = recall_score(y_true, y_hat, zero_division=0)
    out["f1"] = f1_score(y_true, y_hat, zero_division=0)
    out["mcc"] = matthews_corrcoef(y_true, y_hat)
    out["kappa"] = cohen_kappa_score(y_true, y_hat)

    # specificity, npv, fpr, fnr
    out["specificity_tnr"] = tn / (tn + fp + 1e-12)
    out["npv"] = tn / (tn + fn + 1e-12)
    out["fpr"] = fp / (fp + tn + 1e-12)
    out["fnr"] = fn / (fn + tp + 1e-12)

    # likelihood ratios
    out["lr_pos"] = out["recall_tpr"] / (out["fpr"] + 1e-12)
    out["lr_neg"] = out["fnr"] / (out["specificity_tnr"] + 1e-12)

    return out

def full_prob_metrics(y_true, p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return {
        "pr_auc": average_precision_score(y_true, p),
        "roc_auc": roc_auc_score(y_true, p),
        "brier": brier_score_loss(y_true, p),
        "logloss": log_loss(y_true, p),
        "ece_15": expected_calibration_error(y_true, p, n_bins=15),
        "ece_20": expected_calibration_error(y_true, p, n_bins=20),
    }

prob_m = full_prob_metrics(y, p_use)
print("=== PROBABILITY METRICS (threshold-free) ===")
for k,v in prob_m.items():
    print(f"{k:>10s}: {v:.6f}")

=== PROBABILITY METRICS (threshold-free) ===
    pr_auc: 0.174817
   roc_auc: 0.712555
     brier: 0.203157
   logloss: 0.585805
    ece_15: 0.333688
    ece_20: 0.333688


In [16]:
# =========================
# CELL E2: CONFUSION MATRICES @ MULTIPLE OPERATING POINTS
# =========================
from sklearn.metrics import precision_recall_curve

def threshold_topk(p, top_frac=0.10):
    k = int(len(p) * top_frac)
    thr = np.sort(p)[-k]  # threshold that keeps top k
    return float(thr)

thr_f1 = best_threshold_by_f1(y, p_use)
op20 = recall_at_precision(y, p_use, 0.20)
op25 = recall_at_precision(y, p_use, 0.25)
thr_top10 = threshold_topk(p_use, 0.10)

ops = [
    ("Best-F1", thr_f1),
    ("Prec>=0.20 (max recall)", op20["threshold"] if op20 else None),
    ("Prec>=0.25 (max recall)", op25["threshold"] if op25 else None),
    ("Top-10% highest risk", thr_top10),
]

rows = []
for name, thr in ops:
    if thr is None:
        continue
    m = compute_metrics_at_threshold(y, p_use, thr)
    m["operating_point"] = name
    rows.append(m)

df_ops = pd.DataFrame(rows)
display(df_ops[[
    "operating_point","threshold","TP","FP","TN","FN",
    "precision","recall_tpr","specificity_tnr","f1","balanced_accuracy","mcc"
]])

df_ops.to_csv(os.path.join(TAB_DIR, "metrics_operating_points.csv"), index=False)
print("Saved:", os.path.join(TAB_DIR, "metrics_operating_points.csv"))

,operating_point,threshold,TP,FP,TN,FN,precision,recall_tpr,specificity_tnr,f1,balanced_accuracy,mcc
0,Best-F1,0.695659,109,357,3884,253,0.233906,0.301105,0.915822,0.263285,0.608463,0.193582
1,Prec>=0.20 (max recall),0.673360,120,480,3761,242,0.200000,0.331492,0.886819,0.249480,0.609155,0.174540
2,Prec>=0.25 (max recall),0.736276,66,198,4043,296,0.250000,0.182320,0.953313,0.210863,0.567817,0.157021
3,Top-10% highest risk,0.697555,106,354,3887,256,0.230435,0.292818,0.916529,0.257908,0.604673,0.187896


Saved: ./outputs/tables/metrics_operating_points.csv


In [17]:
# =========================
# CELL E4: CURVES + DISTRIBUTIONS
# =========================
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

# ROC
plt.figure()
RocCurveDisplay.from_predictions(y, p_use)
plt.title("ROC Curve (OOF)")
save_fig(os.path.join(FIG_DIR, "roc_curve_oof.png"))

# PR
plt.figure()
PrecisionRecallDisplay.from_predictions(y, p_use)
plt.title("Precision–Recall Curve (OOF)")
save_fig(os.path.join(FIG_DIR, "pr_curve_oof.png"))

# Reliability (already available, but keep a standardized name)
plot_reliability(y, p_use, "Reliability (OOF, calibrated)" if p_use is oof_ts else "Reliability (OOF)",
                 os.path.join(FIG_DIR, "reliability_oof.png"), n_bins=15)

# Score distribution by class
plt.figure()
plt.hist(p_use[y==0], bins=40, alpha=0.7, label="y=0")
plt.hist(p_use[y==1], bins=40, alpha=0.7, label="y=1")
plt.title("Predicted probability distribution by class (OOF)")
plt.xlabel("Predicted P(stroke)")
plt.ylabel("Count")
plt.legend()
save_fig(os.path.join(FIG_DIR, "score_distribution_by_class.png"))

print("Saved ROC/PR/Calib/Distributions to:", FIG_DIR)

Saved ROC/PR/Calib/Distributions to: ./outputs/figures


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

In [18]:
# =========================
# CELL E5: PAPER TABLE (ONE ROW SUMMARY)
# =========================
summary = {}
summary.update({f"prob_{k}": v for k,v in full_prob_metrics(y, p_use).items()})

# add op metrics
for name, thr in ops:
    if thr is None:
        continue
    m = compute_metrics_at_threshold(y, p_use, thr)
    prefix = name.lower().replace(" ", "_").replace(">","ge").replace("%","pct").replace("(", "").replace(")", "")
    for k in ["threshold","precision","recall_tpr","specificity_tnr","f1","balanced_accuracy","mcc","TP","FP","TN","FN"]:
        summary[f"{prefix}_{k}"] = m[k]

paper_df = pd.DataFrame([summary])
display(paper_df.T.head(40))

paper_path = os.path.join(TAB_DIR, "paper_metrics_summary.csv")
paper_df.to_csv(paper_path, index=False)
print("Saved:", paper_path)

,0
prob_pr_auc,0.174817
prob_roc_auc,0.712555
prob_brier,0.203157
prob_logloss,0.585805
prob_ece_15,0.333688
prob_ece_20,0.333688
best-f1_threshold,0.695659
best-f1_precision,0.233906
best-f1_recall_tpr,0.301105
best-f1_specificity_tnr,0.915822


Saved: ./outputs/tables/paper_metrics_summary.csv
